# Overfitting, Underfitting si Diagnosticarea Modelelor - Tema

Bine ai venit la tema despre Diagnosticarea Modelelor.

Una dintre cele mai importante abilitati in machine learning este capacitatea de a diagnostica de ce un model nu performeaza asa cum te astepti. Este prea simplu pentru a surprinde tiparele de baza (underfitting)? Sau doar memoreaza datele de antrenare (overfitting)? Intelegerea compromisului bias-variance este cheia pentru a raspunde la aceste intrebari.

In aceasta tema vei explora diverse instrumente si tehnici de diagnostic pentru a identifica si corecta problemele modelelor. Vei lucra cu learning curves pentru a vizualiza performanta modelului pe masura ce dimensiunea datelor creste, vei analiza efectul complexitatii modelului asupra bias-ului si variatiei si vei aplica regularizare pentru a controla overfitting-ul. De asemenea, vei construi un cadru robust pentru compararea modelelor si vei implementa de la zero un generator de learning curves.

Diagnosticarea corecta a problemelor unui model economiseste timp pentru ca iti directioneaza eforturile eficient - iti arata daca trebuie sa colectezi mai multe date, sa adaugi caracteristici, sa cresti complexitatea modelului sau sa aplici regularizare.

**Ce vei face in aceasta tema**

*   Reprezinti si interpretezi **Learning Curves** pentru a diagnostica bias-ul si varianta.
*   Demonstrezi **Compromisul Bias-Variance** folosind regresie polinomiala.
*   Aplici **Regularizare** (Ridge/Lasso) pentru a controla complexitatea modelului.
*   Analizezi **Varianta Cross-Validation** pentru a identifica modele instabile.
*   Construiesti un **cadru sistematic de comparare a modelelor** cu teste statistice.
*   Implementezi un **generator de Learning Curves** de la zero.

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu exceptia celor in care trebuie sa trimiti solutiile sau cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde sa scrii codul solutiei. **Nu adauga si nu modifica niciun cod in afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar evaluatorul le va ignora, asa ca nu te baza pe celulele create de tine pentru codul de raspuns; foloseste spatiile oferite in notebook.

* Evita folosirea variabilelor globale, cu exceptia situatiilor absolut necesare. Evaluatorul iti testeaza codul intr-un mediu izolat fara sa ruleze toate celulele de la inceput. Din acest motiv, variabilele globale pot sa nu fie disponibile in timpul evaluarii. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salveaza-l mai intai apasand pe iconita de salvare din stanga sus a paginii, apoi apasa pe butonul `Submit assignment` din dreapta sus.
---


## Cuprins

1.  [Introducere in diagnosticarea modelelor](#1)
2.  [Learning Curves](#2)
    *   [Exercitiul 1: Reprezinta si diagnosticheaza Learning Curves](#ex-1)
3.  [Compromisul Bias-Variance](#3)
    *   [Exercitiul 2: Analiza regresiei polinomiale](#ex-2)
4.  [Controlul complexitatii](#4)
    *   [Exercitiul 3: Regularizare pentru overfitting](#ex-3)
5.  [Varianta Cross-Validation](#5)
    *   [Exercitiul 4: Analizeaza stabilitatea CV](#ex-4)
6.  [Cadru de comparare a modelelor](#6)
    *   [Exercitiul 5: Comparare sistematica a modelelor](#ex-5)
7.  [Implementare de la zero](#7)
    *   [Exercitiul 6: Generator de Learning Curve](#ex-6)
8.  [Concluzie](#8)


<a name='1'></a>
## 1 - Introducere in diagnosticarea modelelor

**Diagnosticarea modelului** presupune folosirea unor instrumente cantitative si vizuale pentru a intelege caracteristicile performantei modelului.

### Concepte cheie:

**Bias (Underfitting)**:
*   Eroare cauzata de presupuneri prea simpliste.
*   Modelul nu poate surprinde tiparele de baza.
*   Simptome: eroare mare pe train, eroare mare pe validare.

**Variance (Overfitting)**:
*   Eroare cauzata de sensibilitatea la fluctuatiile din setul de antrenare.
*   Modelul trateaza zgomotul ca semnal.
*   Simptome: eroare mica pe train, eroare mare pe validare (decalaj mare).

**Compromisul Bias-Variance**:
*   Cresterea complexitatii $\rightarrow$ Bias mai mic, Variance mai mare.
*   Scaderea complexitatii $\rightarrow$ Bias mai mare, Variance mai mica.
*   Scop: gasirea "punctului optim" cu eroare totala mica.

### Instrumente de diagnostic:

1.  **Learning Curves**: performanta vs dimensiunea setului de antrenare.
2.  **Validation Curves**: performanta vs hiperparametru (complexitate).
3.  **Cross-Validation**: estimare robusta a performantei.
4.  **Analiza reziduurilor**: analiza erorilor de predictie.


### Importa bibliotecile necesare


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.model_selection import learning_curve, validation_curve, train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.datasets import make_regression, load_digits
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully!")

### Functii ajutatoare


In [ ]:
def plot_curves(train_sizes, train_scores, val_scores, title="Learning Curve"):
    """
    Generic plotting function for learning curves.
    """
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)

    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, val_mean, 'o-', color="g", label="Cross-validation score")

    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color="g")

    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Training examples")
    plt.ylabel("Score")
    plt.legend(loc="best")
    plt.grid(True, alpha=0.3)
    plt.show()

print("Helper functions defined!")

### Incarca setul de date
Vom folosi un set de date sintetic pentru regresie, pentru a vizualiza clar conceptele bias-variance, si setul de date digits pentru sarcini de clasificare.


In [ ]:
# Generate synthetic data for regression
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)
X_reg = X_reg ** 2 + 3 # Add non-linearity

# Load digits dataset for classification
digits = load_digits()
X_cls, y_cls = digits.data, digits.target

print(f"Regression Data: {X_reg.shape}")
print(f"Classification Data: {X_cls.shape}")

<a name='2'></a>
## 2 - Learning Curves

**Learning Curves** reprezinta performanta modelului pe setul de antrenare si pe setul de validare in functie de dimensiunea setului de antrenare. Ele sunt folosite pe scara larga pentru a diagnostica daca un algoritm de invatare sufera de bias mare, variance mare sau ambele.

### Interpretare:

*   **Bias mare (Underfitting)**:
    *   Eroarea pe train si eroarea pe validare converg la valori similare.
    *   De obicei, eroarea este mare (scorul este mic).
    *   Adaugarea mai multor date este putin probabil sa ajute.

*   **Variance mare (Overfitting)**:
    *   Exista un decalaj mare intre eroarea pe train si cea pe validare.
    *   Eroarea pe train este mica (scorul este mare).
    *   Eroarea pe validare este mare (scorul este mic).
    *   Adaugarea mai multor date este probabil sa ajute.


<a name='ex-1'></a>
### Exercitiul 1 - Reprezinta si diagnosticheaza Learning Curves

**Sarcina ta:**

Implementeaza functia `generate_learning_curve` astfel incat:
1.  Sa foloseasca `learning_curve` din sklearn.
2.  Sa calculeze scorurile pe train si test pentru dimensiuni diferite ale setului de antrenare.
3.  Sa returneze dimensiunile, scorurile pe train si scorurile pe test.

<details>
  <summary><b><font color="green">Indicii de cod</font></b></summary>

*   Foloseste: `train_sizes, train_scores, test_scores = learning_curve(...)`
*   Transmite `estimator`, `X`, `y`.
*   Seteaza `cv=kf` (splitter pentru cross-validation).
*   Seteaza `train_sizes=np.linspace(0.1, 1.0, 10)` pentru a verifica 10 puncte.
*   Foloseste `n_jobs=-1` pentru procesare paralela.
</details>


In [ ]:
# GRADED FUNCTION: generate_learning_curve
def generate_learning_curve(estimator, X, y, cv=5):
    """
    Generate learning curve data.

    Args:
        estimator: The model to evaluate
        X: Features
        y: Target
        cv: Cross-validation generator or int

    Returns:
        dict containing train_sizes, train_mean, val_mean
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 5-8 lines)
    
    # Generate array of training set sizes (from 10% to 100%)
    train_sizes_seq = None

    # Compute learning curve using sklearn's learning_curve
    # Hint: scoring='neg_mean_squared_error', n_jobs=-1
    train_sizes, train_scores, val_scores = None
    
    # Calculate means (convert from neg_mean_squared_error to RMSE)
    # Hint: RMSE = sqrt(-mean_score)
    train_mean = None
    val_mean = None
    
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'train_sizes': train_sizes,
        'train_mean': train_mean,
        'val_mean': val_mean,
        'train_scores': train_scores,
        'val_scores': val_scores
    }

In [ ]:
# Test with Linear Regression (Likely Underfitting)
lr_model = LinearRegression()
curve_data_lr = generate_learning_curve(lr_model, X_reg, y_reg)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(curve_data_lr['train_sizes'], curve_data_lr['train_mean'], 'o-', label='Training Error (RMSE)')
plt.plot(curve_data_lr['train_sizes'], curve_data_lr['val_mean'], 'o-', label='Validation Error (RMSE)')
plt.title('Learning Curve: Linear Regression (Simple Model)', fontsize=14)
plt.xlabel('Training Set Size')
plt.ylabel('RMSE')
plt.legend()
plt.grid(True)
plt.show()

# You should see errors converging to a relatively high value (~30-40), indicating underfitting.

In [ ]:
unittests.exercise_1(generate_learning_curve)

<a name='3'></a>
## 3 - Compromisul Bias-Variance

Compromisul dintre bias si variance este controlat de complexitatea modelului.
*   **Complexitate mica**: Bias mare, Variance mica.
*   **Complexitate mare**: Bias mic, Variance mare.

Putem vizualiza acest lucru folosind regresie polinomiala si variind gradul polinomului.


<a name='ex-2'></a>
### Exercitiul 2 - Analiza regresiei polinomiale

**Sarcina ta:**

Implementeaza `evaluate_poly_degrees` astfel incat:
1.  Sa itereze printr-o lista de grade polinomiale.
2.  Sa creeze un Pipeline cu `PolynomialFeatures` + `LinearRegression`.
3.  Sa evalueze MSE folosind cross-validation.
4.  Sa returneze rezultatele pentru comparatie.

<details>
  <summary><b><font color="green">Indicii de cod</font></b></summary>

*   Bucla: `for degree in degrees:`
*   Pipeline: `model = make_pipeline(PolynomialFeatures(degree), LinearRegression())`
*   Scor: `scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')`
*   RMSE: `np.sqrt(-np.mean(scores))`
</details>


In [ ]:
# GRADED FUNCTION: evaluate_poly_degrees
def evaluate_poly_degrees(X, y, degrees):
    """
    Evaluate polynomial regression models with different degrees.

    Args:
        X, y: Dataset
        degrees: List of degrees to test

    Returns:
        List of dictionaries with degree and RMSE
    """
    results = []
    
    ### ÎNCEPUT DE COD AICI ### (≈ 10 lines)
    for degree in degrees:
        # Create pipeline: PolynomialFeatures -> LinearRegression
        model = None
        
        # Calculate CV scores (use neg_mean_squared_error)
        scores = None
        
        # Convert to RMSE (root mean squared error)
        rmse = None
        
        # Append to results
        results.append({
            'degree': degree,
            'rmse': rmse
        })
    ### SFÂRȘIT DE COD AICI ###
    
    return results

In [ ]:
# Evaluate degrees
degrees = [1, 2, 3, 5, 10, 15]
poly_results = evaluate_poly_degrees(X_reg, y_reg, degrees)

# Convert to DF
df_poly = pd.DataFrame(poly_results)
print(df_poly)

# Visualize
plt.figure(figsize=(10, 6))
plt.plot(df_poly['degree'], df_poly['rmse'], 'o-', linewidth=2)
plt.title('RMSE vs Polynomial Degree', fontsize=14)
plt.xlabel('Degree (Complexity)')
plt.ylabel('CV RMSE')
plt.grid(True)
plt.show()

In [ ]:
unittests.exercise_2(evaluate_poly_degrees)

<a name='4'></a>
## 4 - Controlul complexitatii

Cand un model sufera de overfitting (Variance mare), **Regularizarea** este o tehnica cheie pentru a-l corecta. Ea penalizeaza modelele complexe, simplificandu-le efectiv.

*   **Ridge (L2)**: adauga penalizarea $\alpha \sum w_i^2$.
*   **Lasso (L1)**: adauga penalizarea $\alpha \sum |w_i|$.


<a name='ex-3'></a>
### Exercitiul 3 - Regularizare pentru overfitting

**Sarcina ta:**

Implementeaza `test_regularization` astfel incat:
1.  Sa ia un polinom de grad mare (de ex. grad 15) care face overfitting.
2.  Sa aplice regresie Ridge cu valori alpha diferite.
3.  Sa returneze scorurile pentru fiecare alpha.

<details>
  <summary><b><font color="green">Indicii de cod</font></b></summary>

*   Creeaza caracteristicile de baza: `poly = PolynomialFeatures(degree=15)`
*   Itereaza alpha: `for alpha in alphas:`
*   Model: `Ridge(alpha=alpha)` (nu uita sa scalezi datele!)
*   Foloseste Pipeline: `make_pipeline(poly, StandardScaler(), ridge)`
</details>


In [ ]:
# GRADED FUNCTION: test_regularization
def test_regularization(X, y, alphas):
    """
    Test Ridge regularization strength on a high-degree polynomial.

    Args:
        X, y: Dataset
        alphas: List of alpha values

    Returns:
        List of results
    """
    results = []
    
    ### ÎNCEPUT DE COD AICI ### (≈ 10-12 lines)
    for alpha in alphas:
        # Create pipeline: Poly -> Scaling -> Ridge
        # Hint: Use make_pipeline with PolynomialFeatures(degree=15), StandardScaler(), and Ridge(...)
        model = None
        
        # Calculate CV Score (neg_mean_squared_error)
        scores = None
        
        # Calculate RMSE
        rmse = None
        
        # Append results
        results.append({
            'alpha': alpha,
            'rmse': rmse
        })
    ### SFÂRȘIT DE COD AICI ###
    
    return results

In [ ]:
# Test alphas
alphas = [0.0001, 0.01, 1, 10, 100, 1000]
reg_results = test_regularization(X_reg, y_reg, alphas)
df_reg = pd.DataFrame(reg_results)

print(df_reg)

plt.figure(figsize=(10, 6))
plt.semilogx(df_reg['alpha'], df_reg['rmse'], 'o-', color='purple')
plt.title('Effect of Regularization (Ridge) on RMSE', fontsize=14)
plt.xlabel('Alpha (Regularization Strength)')
plt.ylabel('RMSE')
plt.grid(True)
plt.show()

In [ ]:
unittests.exercise_3(test_regularization)

<a name='5'></a>
## 5 - Varianta Cross-Validation

Uneori un model are un scor *mediu* bun, dar este **instabil** - performeaza foarte bine pe unele fold-uri si foarte slab pe altele. Acest lucru indica faptul ca modelul este sensibil la punctele de date specifice din split-uri.

Putem analiza **deviatia standard** a scorurilor de cross-validation.


<a name='ex-4'></a>
### Exercitiul 4 - Analizeaza stabilitatea CV

**Sarcina ta:**

Implementeaza `analyze_cv_stability`:
1.  Ruleaza cross-validation.
2.  Calculeaza scorul mediu SI deviatia standard.
3.  Returneaza ambele valori.

<details>
  <summary><b><font color="green">Indicii de cod</font></b></summary>
  
*   `scores = cross_val_score(...)`
*   `mean = scores.mean()`
*   `std = scores.std()`
</details>


In [ ]:
# GRADED FUNCTION: analyze_cv_stability
def analyze_cv_stability(model, X, y, cv=10):
    """
    Analyze stability of a model using CV variance.

    Args:
        model: Sklearn model
        X, y: Data
        cv: Folds

    Returns:
        mean, std, scores
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 3 lines)
    
    # Run cross-validation
    scores = None
    
    # Calculate mean and standard deviation
    mean_score = None
    std_score = None
    
    ### SFÂRȘIT DE COD AICI ###
    
    return mean_score, std_score, scores

In [ ]:
# Compare Random Forest vs Decision Tree on Digits
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

mean_dt, std_dt, scores_dt = analyze_cv_stability(dt, X_cls, y_cls)
mean_rf, std_rf, scores_rf = analyze_cv_stability(rf, X_cls, y_cls)

print(f"Decision Tree: {mean_dt:.4f} (+/- {std_dt:.4f})")
print(f"Random Forest: {mean_rf:.4f} (+/- {std_rf:.4f})")

plt.figure(figsize=(10, 6))
plt.boxplot([scores_dt, scores_rf], labels=['Decision Tree', 'Random Forest'])
plt.title('Model Stability Comparison', fontsize=14)
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

In [ ]:
unittests.exercise_4(analyze_cv_stability)

<a name='6'></a>
## 6 - Cadru de comparare a modelelor

Compararea sistematica a modelelor necesita examinarea mai multor metrici si verificarea faptului ca diferentele sunt semnificative statistic.


<a name='ex-5'></a>
### Exercitiul 5 - Comparare sistematica a modelelor

**Sarcina ta:**

Implementeaza `compare_models`:
1.  Primeste un dictionar de modele.
2.  Evalueaza fiecare model folosind CV.
3.  Realizeaza un test t fata de un baseline (primul model).
4.  Returneaza un DataFrame cu media, deviatia standard si P-value.

<details>
  <summary><b><font color="green">Indicii de cod</font></b></summary>
  
*   Stocheaza scorurile tuturor modelelor intr-un dict `all_scores`.
*   Scorurile baseline: `baseline_scores = all_scores[baseline_name]`.
*   T-test: `t_stat, p_val = stats.ttest_rel(baseline_scores, current_scores)` (foloseste esantioane pereche/corelate).
</details>


In [ ]:
# GRADED FUNCTION: compare_models
def compare_models(models_dict, X, y, cv=10):
    """
    Systematically compare models with statistical tests.

    Args:
        models_dict: Dict of {name: model}
        X, y: Data

    Returns:
        DataFrame with results
    """
    results = []
    
    ### ÎNCEPUT DE COD AICI ### (≈ 15-20 lines)
    
    # 1. Run CV for all models
    all_scores = {} # Dictionary to store scores for each model
    model_names = list(models_dict.keys())
    baseline_name = model_names[0]
    
    # Iterate through models and get CV scores
    for name, model in models_dict.items():
        all_scores[name] = None 
        
    # 2. Compare against baseline
    baseline_scores = all_scores[baseline_name]
    
    for name in model_names:
        current_scores = None # Get scores for current model
        mean = None
        std = None
        
        # Calculate p-value (paired t-test)
        if name == baseline_name:
            p_value = 1.0
        else:
            # Use stats.ttest_rel for paired samples
            _, p_value = None
            
        results.append({
            'Model': name,
            'Mean Accuracy': mean,
            'Std Dev': std,
            'P-Value (vs Baseline)': p_value
        })
        
    ### SFÂRȘIT DE COD AICI ###
    
    return pd.DataFrame(results)

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

comparison_results = compare_models(models, X_cls, y_cls)
print(comparison_results)

In [ ]:
unittests.exercise_5(compare_models)

<a name='7'></a>
## 7 - Implementare de la zero

Intelegerea modului in care sunt construite learning curves ajuta la clarificarea procesului.


<a name='ex-6'></a>
### Exercitiul 6 - Generator de Learning Curve

**Sarcina ta:**

Implementeaza `learning_curve_scratch` astfel incat:
1.  Sa primeasca un model, X, y.
2.  Sa itereze prin dimensiuni specificate ale esantionului de antrenare.
3.  In fiecare iteratie, sa imparta manual datele (de ex. cu `train_test_split`).
4.  Sa antreneze pe subset si sa evalueze pe setul complet de validare (sau pe setul hold-out).
5.  Recomandat: repeta de K ori pentru fiecare dimensiune pentru estimari stabile (stil Monte Carlo).

Pentru simplitate, vom face o singura validare prin split, dar cu setul de antrenare in crestere.

<details>
  <summary><b><font color="green">Indicii de cod</font></b></summary>
  
*   Pune deoparte setul de test *o singura data* la inceput: `X_train_full, X_test, y_train_full, y_test = train_test_split(...)`
*   Itereaza fractiile `[0.1, ... 1.0]`.
*   Subesantionare: `num_samples = int(frac * len(X_train_full))`
*   Slice: `X_subset = X_train_full[:num_samples]`
*   Antreneaza pe subset, evalueaza pe subset (Train Err) si pe `X_test` (Val Err).
</details>


In [ ]:
# GRADED FUNCTION: learning_curve_scratch
def learning_curve_scratch(estimator, X, y, train_fractions=np.linspace(0.1, 1.0, 10)):
    """
    Generate learning curve data from scratch.

    Args:
        estimator: Model
        X, y: Data
        train_fractions: Fractions of training data to use

    Returns:
        sizes, train_scores, val_scores
    """
    
    ### ÎNCEPUT DE COD AICI ### (≈ 15 lines)
    
    # 1. Hold out a validation set using train_test_split (20% is good)
    X_train_full, X_val, y_train_full, y_val = None, None, None, None
    
    sizes = []
    train_scores = []
    val_scores = []
    
    n_train_total = len(X_train_full)
    
    for frac in train_fractions:
        # Determine subset size (based on fraction)
        n_subset = None
        if n_subset == 0: continue
        
        # Create subset (slice the full training set)
        X_sub = None
        y_sub = None
        
        # Train model on subset. Hint: use sklearn.base.clone to get a fresh model
        # model = clone(estimator)
        # model.fit(...)
        
        # Evaluate on subset (Train score) and validation set (Validation score)
        train_sc = None
        val_sc = None
        
        sizes.append(n_subset)
        train_scores.append(train_sc)
        val_scores.append(val_sc)

    ### SFÂRȘIT DE COD AICI ###
    
    return sizes, train_scores, val_scores

In [ ]:
# Test from scratch implementation
rf_scratch = RandomForestClassifier(random_state=42, n_estimators=10)
sizes, t_sc, v_sc = learning_curve_scratch(rf_scratch, X_cls, y_cls)

plt.figure(figsize=(10, 6))
plt.plot(sizes, t_sc, 'o-', label='Train Accuracy')
plt.plot(sizes, v_sc, 'o-', label='Validation Accuracy')
plt.title('Learning Curve (From Scratch)', fontsize=14)
plt.xlabel('Training Examples')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
unittests.exercise_6(learning_curve_scratch)

<a name='8'></a>
## 8 - Concluzie

**Felicitari!** Ai finalizat tema despre Diagnosticarea Modelelor.

### Idei principale:

1.  **Learning Curves** sunt prima ta linie de aparare.
    *   Convergenta la scor mic $\rightarrow$ **Bias** (Underfitting).
    *   Decalaj mare $\rightarrow$ **Variance** (Overfitting).
2.  **Controlul complexitatii**:
    *   Modelele simple (liniare) au nevoie de mai multe caracteristici / complexitate pentru a corecta bias-ul.
    *   Modelele complexe (polinoame de grad mare, arbori adanci) au nevoie de regularizare / mai multe date pentru a corecta varianta.
3.  **Stabilitatea conteaza**:
    *   Verifica mereu deviatia standard a scorurilor CV. Un split "norocos" te poate induce in eroare.
4.  **Comparatie riguroasa**:
    *   Foloseste teste statistice (precum t-test) pentru a confirma daca Modelul A este *intr-adevar* mai bun decat Modelul B.

Succes cu viitoarele tale diagnostice de model!
